# Assignment NLP-5: Fine-Tuning DistilBERT for POS Tagging & Chunking

## Token Classification using Transformer Models


**Pipeline:** Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

**Model:** `distilbert-base-uncased` (lightweight BERT variant)


In [1]:
# ============================================================
# Install required packages
# ============================================================
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# ============================================================
# Import all necessary libraries
# ============================================================
import pandas as pd
import numpy as np
import torch
import gc
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
from seqeval.metrics import (
    classification_report as seqeval_report,
    precision_score as seq_precision,
    recall_score as seq_recall,
    f1_score as seq_f1
)
from sklearn.metrics import classification_report as sklearn_report
from google.colab import files

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


---
## Task 1: Dataset Selection (10%)

**Dataset:** Annotated Corpus for Named Entity Recognition

This dataset contains sentences annotated with:
- **POS Tags** — Part-of-Speech labels (NNS, IN, VBP, NNP, etc.) for grammar-level tagging
- **Chunk/Entity Tags** — IOB-formatted tags (B-geo, B-gpe, O, etc.) for phrase-level grouping

**Label Types:**
- POS column → Grammar-level token classification (e.g., Noun, Verb, Adjective)
- Tag column → Phrase-level IOB tagging (e.g., B-geo, I-geo, O)

In [3]:
# ============================================================
# Upload the dataset manually
# ============================================================
print("Please upload your CSV dataset file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"\nUploaded file: {filename}")

Please upload your CSV dataset file:


Saving ner_dataset.csv to ner_dataset.csv

Uploaded file: ner_dataset.csv


In [5]:
# ============================================================
# Load and explore the dataset
# ============================================================
# Read CSV with latin-1 encoding to handle special characters
df = pd.read_csv(filename, encoding='latin-1')

# Keep only the first 4 columns and rename them
df = df.iloc[:, :4]
df.columns = ['Sentence #', 'Word', 'POS', 'Tag']

# Forward-fill the Sentence # column (only the first word of each sentence has the label)
df['Sentence #'] = df['Sentence #'].ffill()

# Handle any missing values
df['Word'] = df['Word'].fillna('UNK')
df['POS'] = df['POS'].fillna('NN')
df['Tag'] = df['Tag'].fillna('O')

print(f"Dataset shape: {df.shape}")
print(f"Number of sentences: {df['Sentence #'].nunique()}")
print(f"\nFirst 10 rows:")
df.head(10)

Dataset shape: (1048575, 4)
Number of sentences: 47959

First 10 rows:


,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O
5,Sentence: 1,through,IN,O
6,Sentence: 1,London,NNP,B-geo
7,Sentence: 1,to,TO,O
8,Sentence: 1,protest,VB,O
9,Sentence: 1,the,DT,O


In [6]:
# ============================================================
# Display label categories for both tasks
# ============================================================
unique_pos_tags = sorted(df['POS'].unique())
unique_chunk_tags = sorted(df['Tag'].unique())

print(f"=== POS Tags ({len(unique_pos_tags)} categories) ===")
print(unique_pos_tags)

print(f"\n=== Chunk/Entity Tags ({len(unique_chunk_tags)} categories) ===")
print(unique_chunk_tags)

print(f"\n=== POS Tag Distribution (Top 10) ===")
print(df['POS'].value_counts().head(10))

print(f"\n=== Chunk/Entity Tag Distribution ===")
print(df['Tag'].value_counts())

=== POS Tags (42 categories) ===
['$', ',', '.', ':', ';', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LRB', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'RRB', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']

=== Chunk/Entity Tags (17 categories) ===
['B-art', 'B-eve', 'B-geo', 'B-gpe', 'B-nat', 'B-org', 'B-per', 'B-tim', 'I-art', 'I-eve', 'I-geo', 'I-gpe', 'I-nat', 'I-org', 'I-per', 'I-tim', 'O']

=== POS Tag Distribution (Top 10) ===
POS
NN     145807
NNP    131426
IN     120996
DT      98454
JJ      78412
NNS     75840
.       47831
VBD     39379
,       32757
VBN     32328
Name: count, dtype: int64

=== Chunk/Entity Tag Distribution ===
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe      

---
## Task 2: Data Preprocessing (15%)

**Steps:**
1. Group words into sentences
2. Tokenize using DistilBERT tokenizer
3. Align labels with subword tokens
4. Handle special tokens with -100 (ignored in loss computation)

**Key Challenge:** BERT tokenizer splits words into subwords. We assign the label only to the first subword and use -100 for the rest.

In [7]:
# ============================================================
# Group words into sentences with their corresponding tags
# ============================================================
grouped = df.groupby('Sentence #')
sentences = grouped['Word'].apply(list).tolist()
pos_tags_list = grouped['POS'].apply(list).tolist()
chunk_tags_list = grouped['Tag'].apply(list).tolist()

print(f"Total sentences: {len(sentences)}")
print(f"\nExample sentence: {sentences[0][:10]}...")
print(f"POS tags:         {pos_tags_list[0][:10]}...")
print(f"Chunk tags:       {chunk_tags_list[0][:10]}...")

Total sentences: 47959

Example sentence: ['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the']...
POS tags:         ['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP', 'TO', 'VB', 'DT']...
Chunk tags:       ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O']...


In [8]:
# ============================================================
# Create label-to-id and id-to-label mappings
# ============================================================
# POS label mappings
pos_label2id = {label: idx for idx, label in enumerate(unique_pos_tags)}
pos_id2label = {idx: label for label, idx in pos_label2id.items()}

# Chunk label mappings
chunk_label2id = {label: idx for idx, label in enumerate(unique_chunk_tags)}
chunk_id2label = {idx: label for label, idx in chunk_label2id.items()}

print(f"POS labels ({len(pos_label2id)}): {pos_label2id}")
print(f"\nChunk labels ({len(chunk_label2id)}): {chunk_label2id}")

POS labels (42): {'$': 0, ',': 1, '.': 2, ':': 3, ';': 4, 'CC': 5, 'CD': 6, 'DT': 7, 'EX': 8, 'FW': 9, 'IN': 10, 'JJ': 11, 'JJR': 12, 'JJS': 13, 'LRB': 14, 'MD': 15, 'NN': 16, 'NNP': 17, 'NNPS': 18, 'NNS': 19, 'PDT': 20, 'POS': 21, 'PRP': 22, 'PRP$': 23, 'RB': 24, 'RBR': 25, 'RBS': 26, 'RP': 27, 'RRB': 28, 'TO': 29, 'UH': 30, 'VB': 31, 'VBD': 32, 'VBG': 33, 'VBN': 34, 'VBP': 35, 'VBZ': 36, 'WDT': 37, 'WP': 38, 'WP$': 39, 'WRB': 40, '``': 41}

Chunk labels (17): {'B-art': 0, 'B-eve': 1, 'B-geo': 2, 'B-gpe': 3, 'B-nat': 4, 'B-org': 5, 'B-per': 6, 'B-tim': 7, 'I-art': 8, 'I-eve': 9, 'I-geo': 10, 'I-gpe': 11, 'I-nat': 12, 'I-org': 13, 'I-per': 14, 'I-tim': 15, 'O': 16}


In [9]:
# ============================================================
# Initialize the DistilBERT tokenizer
# ============================================================
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [10]:
# ============================================================
# Function to tokenize sentences and align labels with subword tokens
# ============================================================
def tokenize_and_align_labels(sentences, tags, label2id, max_length=128):
    """
    Tokenizes sentences and aligns labels with BERT subword tokens.

    Label alignment strategy:
    - First subword of a word → gets the word's label
    - Subsequent subwords   → get -100 (ignored by loss function)
    - Special tokens [CLS], [SEP], [PAD] → get -100

    Returns: dict with input_ids, attention_mask, labels
    """
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for sent, tag_seq in zip(sentences, tags):
        # Tokenize the pre-split words
        encoding = tokenizer(
            sent,
            is_split_into_words=True,
            truncation=True,
            max_length=max_length
        )

        # Get word IDs to map subwords back to original words
        word_ids = encoding.word_ids()

        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                # Special token ([CLS], [SEP], [PAD]) → -100
                label_ids.append(-100)
            elif word_id >= len(tag_seq):
                # Safety check for truncated sequences
                label_ids.append(-100)
            elif word_id != prev_word_id:
                # First subword of a new word → assign the actual label
                label_ids.append(label2id[tag_seq[word_id]])
            else:
                # Subsequent subword of the same word → -100
                label_ids.append(-100)
            prev_word_id = word_id

        all_input_ids.append(encoding['input_ids'])
        all_attention_masks.append(encoding['attention_mask'])
        all_labels.append(label_ids)

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_masks,
        'labels': all_labels
    }

print("Tokenization function defined.")

Tokenization function defined.


In [11]:
# ============================================================
# Tokenize and align labels for POS Tagging
# ============================================================
print("Tokenizing for POS tagging...")
pos_tokenized = tokenize_and_align_labels(sentences, pos_tags_list, pos_label2id)

# Create Hugging Face Dataset and split into train/validation
pos_dataset = Dataset.from_dict(pos_tokenized)
pos_split = pos_dataset.train_test_split(test_size=0.2, seed=42)

print(f"POS Training samples: {len(pos_split['train'])}")
print(f"POS Validation samples: {len(pos_split['test'])}")

# Verify the output format
sample = pos_split['train'][0]
print(f"\nSample input_ids length: {len(sample['input_ids'])}")
print(f"Sample attention_mask length: {len(sample['attention_mask'])}")
print(f"Sample labels length: {len(sample['labels'])}")

Tokenizing for POS tagging...
POS Training samples: 38367
POS Validation samples: 9592

Sample input_ids length: 35
Sample attention_mask length: 35
Sample labels length: 35


In [12]:
# ============================================================
# Tokenize and align labels for Chunking
# ============================================================
print("Tokenizing for Chunking...")
chunk_tokenized = tokenize_and_align_labels(sentences, chunk_tags_list, chunk_label2id)

# Create Hugging Face Dataset and split into train/validation
chunk_dataset = Dataset.from_dict(chunk_tokenized)
chunk_split = chunk_dataset.train_test_split(test_size=0.2, seed=42)

print(f"Chunking Training samples: {len(chunk_split['train'])}")
print(f"Chunking Validation samples: {len(chunk_split['test'])}")

Tokenizing for Chunking...
Chunking Training samples: 38367
Chunking Validation samples: 9592


---
## Task 3: Model Setup (15%)

**Model:** DistilBERT (`distilbert-base-uncased`)
- Lightweight variant of BERT (40% smaller, 60% faster)
- Uses `AutoModelForTokenClassification` for token-level predictions
- Requires correct `num_labels` and proper label mappings (`id2label`, `label2id`)

In [13]:
# ============================================================
# Load DistilBERT model for POS Tagging
# ============================================================
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(pos_label2id),
    id2label=pos_id2label,
    label2id=pos_label2id
)
print(f"POS Model loaded with {len(pos_label2id)} labels")
print(f"Model parameters: {sum(p.numel() for p in pos_model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


POS Model loaded with 42 labels
Model parameters: 66,395,178


In [14]:
# ============================================================
# Load DistilBERT model for Chunking
# ============================================================
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(chunk_label2id),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print(f"Chunking Model loaded with {len(chunk_label2id)} labels")
print(f"Model parameters: {sum(p.numel() for p in chunk_model.parameters()):,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Chunking Model loaded with 17 labels
Model parameters: 66,375,953


---
## Task 4: Training (20%)

**Training Configuration:**
- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16
- Weight decay: 0.01
- Evaluation at each epoch
- FP16 mixed precision (when GPU available)

In [15]:
# ============================================================
# Define compute_metrics function using seqeval
# ============================================================
def make_compute_metrics(id2label):
    """
    Creates a compute_metrics function for the Trainer.
    Uses seqeval for sequence-based evaluation metrics.
    Skips -100 labels (special tokens and subwords).
    """
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=2)

        # Convert to label strings, skipping -100 padding
        true_labels = []
        pred_labels = []
        for pred_seq, label_seq in zip(predictions, labels):
            true_seq = []
            pred_seq_str = []
            for p, l in zip(pred_seq, label_seq):
                if l != -100:
                    true_seq.append(id2label[l])
                    pred_seq_str.append(id2label[p])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq_str)

        return {
            'precision': seq_precision(true_labels, pred_labels),
            'recall': seq_recall(true_labels, pred_labels),
            'f1': seq_f1(true_labels, pred_labels),
        }
    return compute_metrics

# Data collator handles dynamic padding of batches
data_collator = DataCollatorForTokenClassification(tokenizer)

print("Metrics function and data collator ready.")

Metrics function and data collator ready.


In [18]:
# ============================================================
# Train POS Tagging Model
# ============================================================
print("=" * 50)
print("Training POS Tagging Model")
print("=" * 50)

pos_training_args = TrainingArguments(
    output_dir='./pos_model_output',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=2,
    logging_steps=500,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

pos_trainer = Trainer(
    model=pos_model,
    args=pos_training_args,
    train_dataset=pos_split['train'],
    eval_dataset=pos_split['test'],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_id2label),
)

# Train the model
pos_train_result = pos_trainer.train()
print("\nPOS Training complete!")
print(f"Training loss: {pos_train_result.training_loss:.4f}")

Training POS Tagging Model


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.075853,0.062025,0.971976,0.973858,0.972916
2,0.052975,0.055047,0.975660,0.976219,0.975940
3,0.040760,0.054452,0.975993,0.976368,0.976181


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



POS Training complete!
Training loss: 0.0898


In [19]:
# ============================================================
# Train Chunking Model
# ============================================================
print("=" * 50)
print("Training Chunking Model")
print("=" * 50)

chunk_training_args = TrainingArguments(
    output_dir='./chunk_model_output',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=2,
    logging_steps=500,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=chunk_split['train'],
    eval_dataset=chunk_split['test'],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_id2label),
)

# Train the model
chunk_train_result = chunk_trainer.train()
print("\nChunking Training complete!")
print(f"Training loss: {chunk_train_result.training_loss:.4f}")

Training Chunking Model


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.119213,0.107559,0.807176,0.818565,0.812831
2,0.094904,0.100192,0.821304,0.822422,0.821863
3,0.077592,0.100299,0.820528,0.829500,0.824989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Chunking Training complete!
Training loss: 0.1117


---
## Task 5: Evaluation (15%)

Evaluate both models using the **seqeval** metric on the validation set.

**Metrics reported:**
- Precision
- Recall
- F1 Score
- Per-label detailed classification report

In [20]:
# ============================================================
# Helper function to get detailed evaluation
# ============================================================
def evaluate_model(trainer, dataset, id2label, task_name):
    """
    Runs prediction on the dataset and prints detailed seqeval report.
    Returns precision, recall, and F1 scores.
    """
    print(f"\n{'='*60}")
    print(f"  Evaluation Results: {task_name}")
    print(f"{'='*60}")

    # Get predictions
    output = trainer.predict(dataset)
    predictions = np.argmax(output.predictions, axis=2)
    labels = output.label_ids

    # Convert numeric predictions to label strings, skipping -100
    true_labels = []
    pred_labels = []
    for pred_seq, label_seq in zip(predictions, labels):
        true_seq = []
        pred_seq_str = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_seq.append(id2label[l])
                pred_seq_str.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_str)

    # Compute overall metrics
    precision = seq_precision(true_labels, pred_labels)
    recall = seq_recall(true_labels, pred_labels)
    f1 = seq_f1(true_labels, pred_labels)

    print(f"\nOverall Precision: {precision:.4f}")
    print(f"Overall Recall:    {recall:.4f}")
    print(f"Overall F1 Score:  {f1:.4f}")

    # Detailed per-label report
    print(f"\nDetailed Classification Report:")
    print(seqeval_report(true_labels, pred_labels))

    return {'precision': precision, 'recall': recall, 'f1': f1}

print("Evaluation function defined.")

Evaluation function defined.


In [21]:
# ============================================================
# Evaluate POS Tagging Model
# ============================================================
pos_metrics = evaluate_model(
    pos_trainer, pos_split['test'], pos_id2label, "POS Tagging"
)


  Evaluation Results: POS Tagging



Overall Precision: 0.9760
Overall Recall:    0.9764
Overall F1 Score:  0.9762

Detailed Classification Report:
              precision    recall  f1-score   support

           B       0.97      0.97      0.97      8143
          BD       0.98      0.98      0.98      7914
          BG       0.98      0.98      0.98      3785
          BN       0.96      0.96      0.96      5945
          BP       0.98      0.98      0.98      3179
          BR       0.85      0.87      0.86       217
          BS       0.92      0.94      0.93        62
          BZ       0.99      0.99      0.99      4964
           C       1.00      1.00      1.00      4681
           D       1.00      1.00      1.00      6110
          DT       0.97      0.98      0.98       764
           H       1.00      0.50      0.67         4
           J       0.95      0.95      0.95     14388
          JR       0.96      0.97      0.96       622
          JS       0.97      0.98      0.98       635
           N       0.97

In [22]:
# ============================================================
# Evaluate Chunking Model
# ============================================================
chunk_metrics = evaluate_model(
    chunk_trainer, chunk_split['test'], chunk_id2label, "Chunking"
)


  Evaluation Results: Chunking



Overall Precision: 0.8205
Overall Recall:    0.8295
Overall F1 Score:  0.8250

Detailed Classification Report:
              precision    recall  f1-score   support

         art       0.44      0.08      0.13       103
         eve       0.43      0.32      0.37        71
         geo       0.84      0.89      0.87      7359
         gpe       0.96      0.94      0.95      3182
         nat       0.28      0.30      0.29        27
         org       0.67      0.63      0.65      3962
         per       0.75      0.79      0.77      3310
         tim       0.88      0.88      0.88      4027

   micro avg       0.82      0.83      0.82     22041
   macro avg       0.66      0.60      0.61     22041
weighted avg       0.82      0.83      0.82     22041



---
## Task 6: Inference (10%)

Load the trained models and predict POS tags and Chunk tags on custom sentences.

In [23]:
# ============================================================
# Inference function for custom sentences
# ============================================================
def predict_sentence(sentence, model, tokenizer, id2label):
    """
    Predicts token labels for a given sentence.
    Handles subword alignment to map predictions back to original words.
    """
    words = sentence.split()

    # Tokenize (without padding, for single inference)
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    # Get word IDs for mapping back to original words
    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=128
    ).word_ids()

    # Move inputs to model's device
    inputs = {k: v.to(model.device) for k, v in encoding.items()}

    # Get predictions
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)[0]

    # Map predictions back to original words (only first subword per word)
    results = []
    prev_word_id = None
    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id != prev_word_id:
            results.append((words[word_id], id2label[predictions[idx].item()]))
        prev_word_id = word_id

    return results

print("Inference function defined.")

Inference function defined.


In [26]:
# ============================================================
# Run inference on custom sentences
# ============================================================
test_sentences = [

    "I am Imam Jamdar from Mudhol",
    "I love learning Artificial Intelligence and Machine learning",
    "I am an Idustrial Engineering and Manadement student of fourth year",
    "Thanks to Innomatics for this internship opportunity"
]

# Get the trained models from the trainers
pos_trained_model = pos_trainer.model
chunk_trained_model = chunk_trainer.model

for sentence in test_sentences:
    print(f"\n{'='*60}")
    print(f"Input: {sentence}")
    print(f"{'='*60}")

    # POS Tagging predictions
    pos_results = predict_sentence(sentence, pos_trained_model, tokenizer, pos_id2label)
    print("\n  POS Tags:")
    print(f"  {'Word':<20} {'POS Tag':<10}")
    print(f"  {'-'*30}")
    for word, tag in pos_results:
        print(f"  {word:<20} {tag:<10}")

    # Chunk Tag predictions
    chunk_results = predict_sentence(sentence, chunk_trained_model, tokenizer, chunk_id2label)
    print("\n  Chunk Tags:")
    print(f"  {'Word':<20} {'Chunk Tag':<10}")
    print(f"  {'-'*30}")
    for word, tag in chunk_results:
        print(f"  {word:<20} {tag:<10}")


Input: I am Imam Jamdar from Mudhol

  POS Tags:
  Word                 POS Tag   
  ------------------------------
  I                    PRP       
  am                   VBP       
  Imam                 NNP       
  Jamdar               NNP       
  from                 IN        
  Mudhol               NNP       

  Chunk Tags:
  Word                 Chunk Tag 
  ------------------------------
  I                    O         
  am                   O         
  Imam                 B-per     
  Jamdar               I-per     
  from                 O         
  Mudhol               B-geo     

Input: I love learning Artificial Intelligence and Machine learning

  POS Tags:
  Word                 POS Tag   
  ------------------------------
  I                    PRP       
  love                 VBP       
  learning             VBG       
  Artificial           JJ        
  Intelligence         NN        
  and                  CC        
  Machine              NN        
  lear

---
## Task 7: Comparison (10%)

Compare POS Tagging and Chunking tasks across multiple dimensions.

In [28]:
# ============================================================
# Comparison of POS Tagging vs Chunking
# ============================================================
print("=" * 70)
print("  COMPARISON: POS Tagging vs Chunking")
print("=" * 70)

# 1. Performance Comparison
print("\n1. PERFORMANCE METRICS COMPARISON")
print(f"{'Metric':<15} {'POS Tagging':<20} {'Chunking':<20}")
print(f"{'-'*55}")
print(f"{'Precision':<15} {pos_metrics['precision']:<20.4f} {chunk_metrics['precision']:<20.4f}")
print(f"{'Recall':<15} {pos_metrics['recall']:<20.4f} {chunk_metrics['recall']:<20.4f}")
print(f"{'F1 Score':<15} {pos_metrics['f1']:<20.4f} {chunk_metrics['f1']:<20.4f}")

# 2. Task Characteristics Comparison
print("\n2. TASK CHARACTERISTICS")
print(f"{'Characteristic':<25} {'POS Tagging':<25} {'Chunking':<25}")
print(f"{'-'*75}")
print(f"{'Tagging Level':<25} {'Token-level (word)':<25} {'Phrase-level (span)':<25}")
print(f"{'Difficulty':<25} {'Easy':<25} {'Medium':<25}")
print(f"{'Number of Labels':<25} {len(pos_label2id):<25} {len(chunk_label2id):<25}")
print(f"{'Tag Format':<25} {'Flat tags (NN, VB)':<25} {'IOB tags (B-geo, I-geo)':<25}")
print(f"{'Purpose':<25} {'Grammar categories':<25} {'Entity/phrase boundaries':<25}")
print(f"{'Context Needed':<25} {'Mostly local':<25} {'Broader context':<25}")

# 3. Key Observations
print("\n3. KEY OBSERVATIONS")
print("-" * 70)

if pos_metrics['f1'] > chunk_metrics['f1']:
    print(f"  • POS Tagging achieved higher F1 ({pos_metrics['f1']:.4f} vs {chunk_metrics['f1']:.4f})")
    print(f"    This is expected as POS tagging is a simpler task (grammar-level).")
else:
    print(f"  • Chunking achieved higher F1 ({chunk_metrics['f1']:.4f} vs {pos_metrics['f1']:.4f})")
    print(f"    This may be due to fewer chunk labels making it easier to learn.")

print(f"  • POS tagging has {len(pos_label2id)} label categories vs {len(chunk_label2id)} for Chunking.")
print(f"  • POS tags assign one label per word (no span grouping needed).")
print(f"  • Chunk tags use IOB format to identify multi-word phrases/entities.")
print(f"  • Both tasks benefit from DistilBERT's contextual word representations.")

  COMPARISON: POS Tagging vs Chunking

1. PERFORMANCE METRICS COMPARISON
Metric          POS Tagging          Chunking            
-------------------------------------------------------
Precision       0.9760               0.8205              
Recall          0.9764               0.8295              
F1 Score        0.9762               0.8250              

2. TASK CHARACTERISTICS
Characteristic            POS Tagging               Chunking                 
---------------------------------------------------------------------------
Tagging Level             Token-level (word)        Phrase-level (span)      
Difficulty                Easy                      Medium                   
Number of Labels          42                        17                       
Tag Format                Flat tags (NN, VB)        IOB tags (B-geo, I-geo)  
Purpose                   Grammar categories        Entity/phrase boundaries 
Context Needed            Mostly local              Broader context   

In [30]:
# ============================================================
# Side-by-side inference comparison
# ============================================================
print("\n4. SIDE-BY-SIDE INFERENCE COMPARISON")
print("=" * 70)

comparison_sentence = "John works at Google in California"
pos_results = predict_sentence(comparison_sentence, pos_trained_model, tokenizer, pos_id2label)
chunk_results = predict_sentence(comparison_sentence, chunk_trained_model, tokenizer, chunk_id2label)

print(f"\nInput: \"{comparison_sentence}\"\n")
print(f"{'Word':<15} {'POS Tag':<15} {'Chunk Tag':<15}")
print(f"{'-'*45}")
for (word, pos_tag), (_, chunk_tag) in zip(pos_results, chunk_results):
    print(f"{word:<15} {pos_tag:<15} {chunk_tag:<15}")

print("\n--- Interpretation ---")
print("POS Tags tell us the grammatical role of each word (Noun, Verb, Preposition, etc.)")
print("Chunk Tags identify named entities/phrases by marking their boundaries (B-tag = Begin, I-tag = Inside, O = Outside)")


4. SIDE-BY-SIDE INFERENCE COMPARISON

Input: "John works at Google in California"

Word            POS Tag         Chunk Tag      
---------------------------------------------
John            NNP             B-per          
works           VBZ             O              
at              IN              O              
Google          NNP             B-org          
in              IN              O              
California      NNP             B-geo          

--- Interpretation ---
POS Tags tell us the grammatical role of each word (Noun, Verb, Preposition, etc.)
Chunk Tags identify named entities/phrases by marking their boundaries (B-tag = Begin, I-tag = Inside, O = Outside)


---
## Task 8: Report / Blog (5%)

### Differences between POS Tagging and Chunking

| Aspect | POS Tagging | Chunking |
|--------|-------------|----------|
| **Level** | Token-level | Phrase-level |
| **Purpose** | Assigns grammatical categories (Noun, Verb, Adjective) | Groups tokens into phrases/entities (Person, Location, Organization) |
| **Tag Format** | Flat labels (NN, VB, JJ) | IOB format (B-geo, I-geo, O) |
| **Difficulty** | Easier — local context often sufficient | Harder — requires understanding phrase boundaries |
| **Output** | One tag per word | Spans of multiple words forming entities |

### Challenges Faced

1. **Subword Tokenization Alignment:** BERT's WordPiece tokenizer splits words into subwords. Aligning labels required careful mapping — only the first subword gets the actual label, while subsequent subwords receive -100 (ignored by the loss function).

2. **Label Imbalance:** The 'O' (Outside) tag dominates the chunk dataset, making it harder for the model to learn minority entity classes. POS tags have a more balanced distribution across categories.

3. **Memory and Compute Constraints:** Training two separate transformer models on Google Colab required efficient use of GPU memory through mixed precision training (FP16) and dynamic padding.

4. **Sequence Length Handling:** Sentences exceeding 128 tokens needed truncation, potentially losing information at the end of long sentences.

### Observations and Insights

1. **DistilBERT's Effectiveness:** Even a distilled (smaller) version of BERT achieves strong performance on both tasks, demonstrating the power of pre-trained language models for token classification.

2. **Transfer Learning Value:** Fine-tuning a pre-trained model requires far less data and training time compared to training from scratch, while achieving superior results.

3. **POS vs Chunking Difficulty:** POS tagging typically achieves higher accuracy because it relies on local context (surrounding words), while chunking requires understanding broader phrase structure and entity boundaries.

4. **IOB Tagging Scheme:** The IOB format is crucial for chunking as it distinguishes between the beginning of an entity (B-tag) and continuation tokens (I-tag), enabling multi-word entity recognition.

5. **Practical Applications:** POS tagging supports downstream NLP tasks like parsing and grammar checking, while chunking/NER supports information extraction, question answering, and knowledge graph construction.

In [47]:
# ============================================================
# Final Summary
# ============================================================
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"\nModel Used: {MODEL_NAME}")
print(f"Dataset: Annotated Corpus ({len(sentences)} sentences)")
print(f"\nPOS Tagging Results:")
print(f"  Precision: {pos_metrics['precision']:.4f}")
print(f"  Recall:    {pos_metrics['recall']:.4f}")
print(f"  F1 Score:  {pos_metrics['f1']:.4f}")
print(f"\nChunking Results:")
print(f"  Precision: {chunk_metrics['precision']:.4f}")
print(f"  Recall:    {chunk_metrics['recall']:.4f}")
print(f"  F1 Score:  {chunk_metrics['f1']:.4f}")
print(f"\nPipeline: Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison")
print("\n✅ Assignment NLP-5 Complete!")

  FINAL SUMMARY

Model Used: distilbert-base-uncased
Dataset: Annotated Corpus (47959 sentences)

POS Tagging Results:
  Precision: 0.9760
  Recall:    0.9764
  F1 Score:  0.9762

Chunking Results:
  Precision: 0.8205
  Recall:    0.8295
  F1 Score:  0.8250

Pipeline: Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

✅ Assignment NLP-5 Complete!
